# Assignment 5: Two-Asset PDE and Monte Carlo Simulation

### a) Fair Value Formula

Using the identity $\max(S_1, S_2) = S_1 + \max(S_2 - S_1, 0)$, the contract decomposes into asset $S_1$ plus an exchange option (Margrabe's formula).

With $\sigma_1 = \sigma_2 = \sigma$ and $\rho = 0$, the volatility of the ratio is $\sigma_d = \sqrt{\sigma^2 + \sigma^2} = \sqrt{2}\sigma$.

The fair value is:
$$V(0) = S_1(0)\Phi(d_1) + S_2(0)\Phi(d_2)$$
where $d_1 = \frac{\sigma\sqrt{T}}{\sqrt{2}}$ and $d_2 = -d_1$.

Since $S_1(0) = S_2(0) = 100$ and $\Phi(-d_1) = 1 - \Phi(d_1)$:
$$V(0) = 200 \cdot \Phi\left(\frac{\sigma \sqrt{T}}{\sqrt{2}}\right)$$

### b) PDE Derivation

Under risk-neutral dynamics ($dS_i = rS_i dt + \sigma S_i dW_i$, $dW_1 dW_2 = 0$), the 2D Black-Scholes PDE is:
$$\frac{\partial V}{\partial t} + r S_1 \frac{\partial V}{\partial S_1} + r S_2 \frac{\partial V}{\partial S_2} + \frac{1}{2}\sigma^2 S_1^2 \frac{\partial^2 V}{\partial S_1^2} + \frac{1}{2}\sigma^2 S_2^2 \frac{\partial^2 V}{\partial S_2^2} - r V = 0$$
Terminal condition: $V(S_1, S_2, T) = \max(S_1, S_2)$.

**Dimension Reduction:**
Using the log-ratio $y = \ln(S_1 / S_2)$ and $V(S_1, S_2, t) = S_2 \cdot u(y, t)$, the PDE simplifies with $\sigma_d = \sqrt{2}\sigma$ and $\tau = T - t$ to:
$$\frac{\partial u}{\partial \tau} = \frac{1}{2}\sigma_d^2 \left( \frac{\partial^2 u}{\partial y^2} - \frac{\partial u}{\partial y} \right)$$
Terminal condition: $u(y, 0) = \max(e^y, 1)$.

### c) & d) Numerical Results: PDE vs. Monte Carlo vs. Analytic

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import norm

# Parameters
S1_0 = 100.0
S2_0 = 100.0
sigma = 0.20
r = 0.05
T = 1.0

# 1. Analytic
d1 = (sigma * np.sqrt(T)) / np.sqrt(2)
price_analytic = 200.0 * norm.cdf(d1)

# 2. Monte Carlo
N_sim = 200000
np.random.seed(42)
Z1 = np.random.normal(0, 1, N_sim)
Z2 = np.random.normal(0, 1, N_sim)
S1_T = S1_0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z1)
S2_T = S2_0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z2)
price_mc = np.exp(-r * T) * np.mean(np.maximum(S1_T, S2_T))

# 3. PDE (Finite Difference)
sigma_d = np.sqrt(2) * sigma
Ny, Nt = 200, 1000
y_min, y_max = -2.0, 2.0
dy = (y_max - y_min) / Ny
dt = T / Nt

y = np.linspace(y_min, y_max, Ny + 1)
u = np.maximum(np.exp(y), 1.0)

for n in range(Nt):
    u_yy = (u[2:] - 2 * u[1:-1] + u[:-2]) / (dy**2)
    u_y = (u[2:] - u[:-2]) / (2 * dy)
    u[1:-1] += dt * 0.5 * (sigma_d**2) * (u_yy - u_y)
    u[0] = 1.0
    u[-1] = np.exp(y_max)

idx_0 = np.argmin(np.abs(y - 0.0))
price_pde = S2_0 * u[idx_0]

# Comparison Table
df_res = pd.DataFrame({
    'Method': ['Analytical (Margrabe)', 'Monte Carlo', 'Finite Difference (PDE)'],
    'Fair Value': [price_analytic, price_mc, price_pde],
    'Absolute Error': [0.0, abs(price_mc - price_analytic), abs(price_pde - price_analytic)]
})
df_res

,Method,Fair Value,Absolute Error
0,Analytical (Margrabe),111.246292,0.000000
1,Monte Carlo,111.258962,0.012670
2,Finite Difference (PDE),111.240423,0.005869
